# Export a training set for BiaPy or DL4MicEverywhere

Downloads the annotated units of a training set from OMERO into the folder layout that
[BiaPy](https://biapy.readthedocs.io) and [DL4MicEverywhere](https://github.com/HenriquesLab/DL4MicEverywhere)
expect, then shows how to start training there. This package only prepares the data; the
training itself runs in those tools.

Images keep their native bit depth. How to normalise them is recorded in
`normalization.json` next to the data (see the BiaPy section).

## 1. Connect to OMERO

In [ ]:
from pathlib import Path

from omero_annotate_ai import (
    create_omero_connection_widget,
    create_training_data_widget,
    prepare_training_data_from_table,
)

conn_widget = create_omero_connection_widget()
conn_widget.display()

In [ ]:
conn = conn_widget.get_connection()
if conn is None:
    raise ConnectionError("Not connected yet: use the widget above, then rerun this cell.")
print("Connected to OMERO.")

## 2. Choose the training set

In [ ]:
training_widget = create_training_data_widget(connection=conn)
training_widget.display()

In [ ]:
table_id = training_widget.get_selected_table_id()
if not table_id:
    raise ValueError("Select a training set above, then rerun this cell.")
table_name = training_widget.get_selected_table_info().get("name", f"table_{table_id}")
print(f"Training set: {table_name} (table {table_id})")

## 3. Download the data

Each annotated unit goes to `train_*`, `val_*` or `test_*` according to its split. Change
`output_directory` if you want the data somewhere else.

In [ ]:
output_directory = Path.home() / "omero_annotate_ai_exports" / f"{table_name}_{table_id}"

training_result = prepare_training_data_from_table(
    conn=conn,
    table_id=table_id,
    output_dir=output_directory,
    training_name=table_name,
    clean_existing=True,
)

for key, value in training_result["stats"].items():
    print(f"{key}: {value}")
print(f"Data folder: {output_directory}")

## 4a. BiaPy

Point BiaPy's `DATA.TRAIN.PATH` / `DATA.TRAIN.GT_PATH` (and the validation/test paths) at the
folders above. `biapy_normalization` turns the recorded normalisation into BiaPy's
`DATA.NORMALIZATION` block.

In [ ]:
import json

from omero_annotate_ai.core.annotation_config import NormalizationConfig
from omero_annotate_ai.processing.normalization import biapy_normalization

recorded = json.loads((output_directory / "normalization.json").read_text())
recorded["apply_at_export"] = recorded.pop("applied_at_export", False)
spec = NormalizationConfig.model_validate(
    {k: v for k, v in recorded.items() if k in NormalizationConfig.model_fields}
)
print("DATA.NORMALIZATION:", biapy_normalization(spec))
for key in ("train_input", "train_label", "val_input", "val_label"):
    print(f"{key}: {training_result[key]}")

## 4b. DL4MicEverywhere

Set the path to your DL4MicEverywhere checkout and the notebook you want to train with, then
run the printed command in a terminal.

In [ ]:
dl4mic_folder = Path.home() / "DL4MicEverywhere"  # your checkout
dl4mic_notebook = "Cellpose_2D_DL4Mic"  # any ZeroCostDL4Mic notebook
with_gpu = True

(output_directory / "output").mkdir(parents=True, exist_ok=True)
gpu_flag = " -g" if with_gpu else ""
print(f"cd {dl4mic_folder}")
print(
    f"bash Linux_launch.sh -c notebooks/ZeroCostDL4Mic_notebooks/{dl4mic_notebook}/configuration.yaml "
    f"-d {output_directory} -o {output_directory / 'output'}{gpu_flag}"
)

In [ ]:
conn.close()